In [2]:
import pandas as pd
import os
import re
import json

df = pd.read_csv('../data/evaluations.csv', sep='\t')
df.head()

,ParticipantID,Paper1,Paper2,Paper3,Paper4,Paper5,Paper6,Paper7,Paper8,Paper9,...,Expertise1,Expertise2,Expertise3,Expertise4,Expertise5,Expertise6,Expertise7,Expertise8,Expertise9,Expertise10
0,1737249,4264599665522594d9ecb521dd2e1d002e85a961,c50f98961c951fe3fbdb6f375beb28e40a6b0581,2b4edb9515a26561ea3f9ee2a63a506721c8369e,eba4c6b0b860a34461ffb8544111c89a3ef0d8b7,e54a4e49917eb3da18c2f239be70a68fbd3274c3,daa7e6af585d03e9cb05487413a6495f23400398,22d2f8030221bd0c27bfb9416eeffe4e86633780,3c0d4dc4237934e37467f4ede3af859bcb140abf,5f563da2843e005c4b236f7889e7a22631b53210,...,5.0,4.50,2.5,4.25,2.0,1.00,1.75,1.00,4.75,4.25
1,1700325,3db9649f2ae986cac13f3e748375f8802f9b07fc,429b65937d4922578a81e1f0ef5aeab7361ae36b,9ce09b03f056253252f3e8c0c65d86a27117a0ac,6b2b5d3d9a2ca4bc4fbd81551a62370be2fbff1b,803c7fdd6e01e1ff8cd43297f4e052078409456d,b19cba7bfe318c69d5e62f8322cb5d75228452f4,48b18bf5c9cad0e4c36b2d885f380c5c637e1a09,77568c594470f9aa029f92774e2c12ab0451d9bb,48530f3d6425f2f150f07ccdd61ba951951a0a7d,...,5.0,4.50,2.5,1.50,2.0,3.50,2.00,3.25,4.75,4.00
2,31211315,9045bf2a9c1e2b9621c69c57f991d10880e91f18,63e7e3b16e03da62a2c535ac9cfccfa3ae48b292,1109f787fc8d51feb3bae9bf6e1945dc4a1191e7,d2a2be6ce932a0f1939f31cfff4d64ea3d76723d,e12c52fb542f76b3f0d29178842428d6a4edfe1e,35ee53492c7f32dbb3b4ed7ba4d1395218b13ee9,6e7e095f46deb297713dcde05991faf635768d29,2826ac3621fdd599303c97cb9e32f165521967b2,a445adf335aa5212f929f67c1ca56a62c221b43a,...,1.0,4.75,5.0,3.25,4.0,4.75,2.00,3.75,4.50,2.75
3,2067056951,5aea95e1ae78a66474051a330ded374e199b658c,ef9ddbc35676ce8ffc2a8067044473727839dbac,a38e0f993e4805ba8a9beae4c275c91ffcec01df,03006aefccdd0c5c6736ab11ed574d02ba1cc086,73fe797b4f4f2d18784246bb74626426a8fe108e,bb6317bbd2c4a81e94cf3d7eb1b73da246a022db,0c47eb31b2dd76d8dc986173a1d3f00da1c9c74d,1ccd031f28dccfb226f6c0c588c93a97a50bf95f,0735fb79bf34698c1df4461a05ed51c232c412e4,...,5.0,3.50,5.0,2.50,5.0,4.00,4.00,4.75,3.50,5.00
4,40027632,bb6317bbd2c4a81e94cf3d7eb1b73da246a022db,acbdbf49f9bc3f151b93d9ca9a06009f4f6eb269,0c47eb31b2dd76d8dc986173a1d3f00da1c9c74d,2406cf39805c70264c4226b7325a09b506c70921,d170bd486e4c0fe82601e322b0e9e0dde63ab299,ef9ddbc35676ce8ffc2a8067044473727839dbac,NaN,NaN,NaN,...,5.0,4.25,5.0,4.00,4.5,4.00,NaN,NaN,NaN,NaN


### Query LLMs

In [3]:
import openai
from openai import OpenAI
os.environ["OPENAI_API_KEY"] = "" # add your openai api key here
openai.api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI()
models = client.models.list()

def get_openai_completion(prompt, model="o1-mini"):
   messages = [{"role": "user", "content": prompt}]
   response = client.chat.completions.create(
      model=model,
      messages=messages,
      temperature=1, # this is the degree of randomness of the model's output
   )
   return response.choices[0].message.content

ModuleNotFoundError: No module named 'openai'

In [23]:
import anthropic

client = anthropic.Anthropic(
  # add your anthropic api key here
  api_key="",
)
def get_anthropic_completion(prompt, model='claude-3-5-sonnet-20241022'):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return response.content[0].text

In [11]:
from google import genai

client = genai.Client(api_key="") # add your gemini api key here
def get_gemini_completion(prompt, model="gemini-2.0-flash"):
    response = client.models.generate_content(
        model=model, contents=prompt)
    return response.text

In [ ]:
cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'
prompt_file_0shot = 'prompt_0shot.txt'
with open(f'{data_dir}/{prompt_file_0shot}', 'r') as f:
    prompt = f.read()

not_exact_outputs = []
for filename in [f'd_20_{i}' for i in range(1, 11)]:
    not_exact_output = 0
    filename_path = os.path.join(data_dir, filename)
    reviewer_folder = filename_path + '/archives/'
    
    if os.path.exists(f'../predictions/gemini2flash_{filename}_ta.json'):
        score_dic = json.load(open(f'../predictions/gemini2flash_{filename}_ta.json'))
    else:
        score_dic = {}
    
    for file in os.scandir(reviewer_folder):
        reviewer_dic = {}
        reviewer = int(file.name.strip('.jsonl')[1:])
        
        if str(reviewer) in score_dic:
            print('Already done for reviewer:', reviewer)
            continue
        with open(f'{filename_path}/reviewers/{reviewer}.txt', 'r') as f:
            reviewer_content = f.read() 
        
        row = df[df['ParticipantID'] == reviewer].iloc[0]
        reviewer_expertise_papers =  {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                            if not pd.isna(row[f'Paper{x}'])}
        for p in list(reviewer_expertise_papers.keys()):
            
            with open(f'{filename_path}/papers/{p}.txt', 'r') as f:
                paper_content = f.read()
                full_prompt = f'{prompt}\n<<\nInput: \n{reviewer_content}=====================\n{paper_content}>>'
                
                with open(f'{data_dir}/test_prompt.txt', 'w') as f:
                    f.write(full_prompt)
            
            with open(f'{data_dir}/results.txt', 'a') as f:
                # change the completion function to use different models
                results = get_gemini_completion(full_prompt, model="gemini-2.0-flash")
                f.write(f'{reviewer}, {p}\n')
                f.write(results)
                f.write('\n')
                f.write('=====================\n')
                try:
                    score = float(results.split('\n')[0])
                except:
                    score = float(re.sub(r'[^\d]+', '', results.split('\n')[0]))
                    not_exact_output += 1
                reviewer_dic[p] = score
        
        print('Completed for reviewer:', reviewer)
        score_dic[str(reviewer)] = reviewer_dic
        with open(f'../predictions/gemini2flash_{filename}_ta.json', 'w') as f:
            json.dump(score_dic, f)
    
    with open(f'../predictions/gemini2flash_{filename}_ta.json', 'w') as f:
        json.dump(score_dic, f)
    print('Completed for dataset:', filename)
    not_exact_outputs.append(not_exact_output)

print('Not exact output:', not_exact_outputs)

In [4]:
participants = set(df['ParticipantID'])

# Getting papers
papers = set()
for x in range(1, 11):
    tmp_papers = set(df[~pd.isna(df[f'Paper{x}'])][f'Paper{x}'])
    papers = papers.union(tmp_papers)
print('# of Papers', len(papers))

# Translating df from csv to dict of the form {participant: {Paper1: Expertise1, Paper2: Expertise2, ...}}
data = {}
for idx, row in df.iterrows():
    key = str(row['ParticipantID'])
    data[key] = {row[f'Paper{x}']: row[f'Expertise{x}'] for x in range(1, 11)
                                                            if not pd.isna(row[f'Paper{x}'])}
    
rev_profiles = {}
for rev in participants:
    with open(f'../data/participants/{rev}.json', 'r') as handler:
        rev_profiles[rev] = json.load(handler)
        rev_profiles[rev]['papers'] = set([rev_profiles[rev]['papers'][i]['paperId'] for i in range(len(rev_profiles[rev]['papers']))])
print('# of Participants', len(rev_profiles))

# of Papers 463
# of Participants 58


### SPECTER 2

In [5]:
from transformers import AutoTokenizer
from adapters import AutoAdapterModel

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')

#load base model
model = AutoAdapterModel.from_pretrained('allenai/specter2_base')

#load the adapter(s) as per the required task, provide an identifier for the adapter in load_as argument and activate it
model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

papers = [{'title': 'BERT', 'abstract': 'We introduce a new language representation model called BERT'},
          {'title': 'Attention is all you need', 'abstract': ' The dominant sequence transduction models are based on complex recurrent or convolutional neural networks'}]

# concatenate title and abstract
text_batch = [d['title'] + tokenizer.sep_token + (d.get('abstract') or '') for d in papers]
# preprocess the input
inputs = tokenizer(text_batch, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=512)
output = model(**inputs)
# take the first token in the batch as the embedding
embeddings = output.last_hidden_state[:, 0, :]

ModuleNotFoundError: No module named 'adapters'

In [ ]:
import os 
import json
import pandas as pd

from transformers import AutoTokenizer
from adapters import AutoAdapterModel
from sklearn.metrics.pairwise import cosine_similarity

# load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('allenai/specter2_base')

#load base model
model = AutoAdapterModel.from_pretrained('allenai/specter2_base')

#load the adapter(s) as per the required task, provide an identifier for the adapter in load_as argument and activate it
model.load_adapter("allenai/specter2", source="hf", load_as="specter2", set_active=True)

cur_dir = os.getcwd()
dic = {}
data_dir = '../evaluation_datasets/'

df = pd.read_csv('../data/evaluations.csv', sep='\t')
# get the papers
for filename in [f'd_20_{i}' for i in range(1, 11)]:
    filename_path = os.path.join(data_dir, filename)
    reviewer_folder = filename_path + '/archives/'
    paper_file = filename_path + '/submissions.json'
    
    if os.path.exists(f'../predictions/spector2_all_{filename}_ta.json'):
        score_dic = json.load(open(f'../predictions/spector2_all_{filename}_ta.json'))
    else:
        score_dic = {}
    if not os.path.exists(paper_file):
        continue
    # all paper content
    paper_content = json.load(open(paper_file))
    
    # get paper embeddings
    papers = list(paper_content.keys())
    paper_ta = {p: {'title': paper_content[p]['content']['title'], 'abstract':paper_content[p]['content']['abstract']} for p in papers}
    
    for file in os.scandir(reviewer_folder):
        
        reviewer = int(file.name.strip('.jsonl')[1:])
        if str(reviewer) in score_dic:
            print('Already done for reviewer:', reviewer)
            continue
        with open(file) as f:
            reviewer_published_papers = [json.loads(line) for line in f]
            published_paper_content = {p['id']: {'title': p['content']['title'], 'abstract': p['content']['abstract']} for p in reviewer_published_papers}
            
        row = df[df['ParticipantID'] == reviewer].iloc[0]
        reviewer_expertise_papers =  [row[f'Paper{x}'] for x in range(1, 11)if not pd.isna(row[f'Paper{x}'])]
        expertise_paper_content = {p: {'title': paper_content[p]['content']['title'], 'abstract':paper_content[p]['content']['abstract']} for p in reviewer_expertise_papers}

        p_batch = [d['title'] + tokenizer.sep_token + (d.get('abstract') or '') for d in published_paper_content.values()]
        e_batch = [d['title'] + tokenizer.sep_token + (d.get('abstract') or '') for d in expertise_paper_content.values()]

        expertise_input = tokenizer(e_batch, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=512)
        e_output = model(**expertise_input)
        e_embeddings = e_output.last_hidden_state[:, 0, :]

        published_input = tokenizer(p_batch, padding=True, truncation=True,
                                   return_tensors="pt", return_token_type_ids=False, max_length=512)
        p_output = model(**published_input)
        p_embeddings = p_output.last_hidden_state[:, 0, :]

        # get cosine simliarity
        sim = cosine_similarity(e_embeddings.detach().numpy(), p_embeddings.detach().numpy())
        reviewer_dic = {str(p): [float(j) for j in s] for p, s in zip(expertise_paper_content.keys(), sim)}
        score_dic[str(reviewer)] = reviewer_dic

        print('Completed for reviewer:', reviewer)
        score_dic[str(reviewer)] = reviewer_dic
        with open(f'../predictions/spector2_all_{filename}_ta.json', 'w') as f:
            json.dump(score_dic, f)
    with open(f'../predictions/spector2_all_{filename}_ta.json', 'w') as f:
        json.dump(score_dic, f)
    print('Completed for dataset:', filename)